In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

db_path = Path("chroma_db")
chroma_client = chromadb.PersistentClient(path=str(db_path))

resumes_path = Path("resumes")
resume_files = [f for f in resumes_path.iterdir() if f.is_file()]

print(f"Client OpenAI pronto. Trovati {len(resume_files)} file nella cartella resumes:")
for f in resume_files:
    print(f" - {f.name}")
    
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

collection = chroma_client.get_or_create_collection(name="hr_resumes")

for f in resume_files:
    if f.suffix.lower() == ".txt":
        with open(f, "r", encoding="utf-8") as file:
            content = file.read()
            
            chunks = chunk_text(content)
            
            for idx, chunk in enumerate(chunks):
                chunk_id = f"{f.name}_chunk_{idx}"
                collection.add(
                    documents=[chunk],
                    ids=[chunk_id],
                    metadatas=[{"source": f.name}]
                )

print(f"Database vettoriale popolato! Totale elementi nella collection: {collection.count()}")

Client OpenAI pronto. Trovati 8 file nella cartella resumes:
 - CV1.txt
 - CV2.txt
 - CV3.txt
 - CV4.txt
 - CV5.txt
 - CV6.txt
 - CV7.txt
 - CV8.txt


ModuleNotFoundError: No module named 'langchain_text_splitters'